In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
# ============================================================
# Store Sales - Time Series Forecasting
# Refined Professional LightGBM Pipeline
# ============================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
import lightgbm as lgb


# ============================================================
# Config
# ============================================================

SEED = 42
VALID_DAYS = 16

LAGS = [16, 17, 18, 21, 28, 35, 42, 56, 63, 91, 182, 364]
ROLL_WINDOWS = [7, 14, 28, 56]


# ============================================================
# Auto Detect Kaggle Data Path
# ============================================================

possible_dirs = [
    "/kaggle/input/store-sales-time-series-forecasting",
    "/kaggle/input/competitions/store-sales-time-series-forecasting",
]

DATA_DIR = None

for path in possible_dirs:
    if os.path.exists(os.path.join(path, "train.csv")):
        DATA_DIR = path
        break

if DATA_DIR is None:
    print("Available input files:")
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
    raise FileNotFoundError("Could not find train.csv. Check Kaggle input path.")

print("Using data directory:", DATA_DIR)

TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
STORES_PATH = f"{DATA_DIR}/stores.csv"
OIL_PATH = f"{DATA_DIR}/oil.csv"
HOLIDAYS_PATH = f"{DATA_DIR}/holidays_events.csv"
TRANSACTIONS_PATH = f"{DATA_DIR}/transactions.csv"
SAMPLE_PATH = f"{DATA_DIR}/sample_submission.csv"


# ============================================================
# Helpers
# ============================================================

def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))


def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "float64":
            df[col] = df[col].astype("float32")

        elif col_type == "int64":
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= 0:
                if c_max < 255:
                    df[col] = df[col].astype("uint8")
                elif c_max < 65535:
                    df[col] = df[col].astype("uint16")
                elif c_max < 4294967295:
                    df[col] = df[col].astype("uint32")
            else:
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype("int16")
                else:
                    df[col] = df[col].astype("int32")

    return df


# ============================================================
# Load Data
# ============================================================

train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test = pd.read_csv(TEST_PATH, parse_dates=["date"])
stores = pd.read_csv(STORES_PATH)
oil = pd.read_csv(OIL_PATH, parse_dates=["date"])
holidays = pd.read_csv(HOLIDAYS_PATH, parse_dates=["date"])
transactions = pd.read_csv(TRANSACTIONS_PATH, parse_dates=["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train["is_test"] = 0
test["is_test"] = 1
test["sales"] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)

del train, test
gc.collect()


# ============================================================
# Base Joins
# ============================================================

df = df.merge(stores, on="store_nbr", how="left")

oil = oil.sort_values("date")
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate().ffill().bfill()
df = df.merge(oil, on="date", how="left")
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

transactions = transactions.rename(columns={"transactions": "store_transactions"})
df = df.merge(transactions, on=["date", "store_nbr"], how="left")
df["store_transactions"] = df["store_transactions"].fillna(0)


# ============================================================
# Holiday Features
# ============================================================

hol = holidays.copy()
hol = hol[hol["transferred"] == False]

national = (
    hol[hol["locale"] == "National"]
    .groupby("date", as_index=False)
    .agg(holiday_national_type=("type", "first"))
)

regional = (
    hol[hol["locale"] == "Regional"]
    .groupby(["date", "locale_name"], as_index=False)
    .agg(holiday_regional_type=("type", "first"))
    .rename(columns={"locale_name": "state"})
)

local = (
    hol[hol["locale"] == "Local"]
    .groupby(["date", "locale_name"], as_index=False)
    .agg(holiday_local_type=("type", "first"))
    .rename(columns={"locale_name": "city"})
)

df = df.merge(national, on="date", how="left")
df = df.merge(regional, on=["date", "state"], how="left")
df = df.merge(local, on=["date", "city"], how="left")

for col in ["holiday_national_type", "holiday_regional_type", "holiday_local_type"]:
    df[col] = df[col].fillna("None")
    df[f"is_{col}"] = (df[col] != "None").astype("int8")


# ============================================================
# Date Features
# ============================================================

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
df["quarter"] = df["date"].dt.quarter
df["dayofyear"] = df["date"].dt.dayofyear

df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype("int8")
df["is_month_start"] = df["date"].dt.is_month_start.astype("int8")
df["is_month_end"] = df["date"].dt.is_month_end.astype("int8")
df["payday"] = ((df["day"] == 15) | (df["is_month_end"] == 1)).astype("int8")

df["earthquake_period"] = (
    (df["date"] >= "2016-04-16") & (df["date"] <= "2016-05-16")
).astype("int8")

df["christmas"] = ((df["month"] == 12) & (df["day"] == 25)).astype("int8")
df["new_year"] = ((df["month"] == 1) & (df["day"] == 1)).astype("int8")


# ============================================================
# Time Series Lag Features
# Lags are >= 16 because test horizon is 16 days.
# ============================================================

df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

group_cols = ["store_nbr", "family"]

for lag in LAGS:
    df[f"sales_lag_{lag}"] = df.groupby(group_cols)["sales"].shift(lag)

for window in ROLL_WINDOWS:
    shifted_sales = df.groupby(group_cols)["sales"].shift(16)

    df[f"sales_roll_mean_{window}"] = (
        shifted_sales
        .groupby([df["store_nbr"], df["family"]])
        .rolling(window)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    df[f"sales_roll_std_{window}"] = (
        shifted_sales
        .groupby([df["store_nbr"], df["family"]])
        .rolling(window)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

    df[f"sales_roll_median_{window}"] = (
        shifted_sales
        .groupby([df["store_nbr"], df["family"]])
        .rolling(window)
        .median()
        .reset_index(level=[0, 1], drop=True)
    )

df["promo_lag_16"] = df.groupby(group_cols)["onpromotion"].shift(16)

df["promo_roll_mean_28"] = (
    df.groupby(group_cols)["onpromotion"]
    .shift(16)
    .groupby([df["store_nbr"], df["family"]])
    .rolling(28)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)


# ============================================================
# Missing Values
# ============================================================

lag_cols = [c for c in df.columns if "sales_lag_" in c or "sales_roll_" in c]
promo_cols = [c for c in df.columns if "promo_" in c]

for col in lag_cols:
    df[col] = df[col].fillna(0)

for col in promo_cols:
    df[col] = df[col].fillna(0)

df["store_transactions"] = df["store_transactions"].fillna(0)
df["dcoilwtico"] = df["dcoilwtico"].fillna(df["dcoilwtico"].median())

df = reduce_mem(df)


# ============================================================
# Categorical Encoding
# ============================================================

cat_cols = [
    "family",
    "city",
    "state",
    "type",
    "cluster",
    "holiday_national_type",
    "holiday_regional_type",
    "holiday_local_type",
]

for col in cat_cols:
    df[col] = df[col].astype("category")


# ============================================================
# Split Data
# ============================================================

max_train_date = df.loc[df["is_test"] == 0, "date"].max()
valid_start = max_train_date - pd.Timedelta(days=VALID_DAYS - 1)

print("Max train date:", max_train_date)
print("Validation start:", valid_start)

train_df = df[(df["is_test"] == 0) & (df["date"] < valid_start)].copy()
valid_df = df[(df["is_test"] == 0) & (df["date"] >= valid_start)].copy()
test_df = df[df["is_test"] == 1].copy()

drop_cols = ["id", "date", "sales", "is_test"]
features = [c for c in df.columns if c not in drop_cols]

X_train = train_df[features]
y_train = np.log1p(train_df["sales"])

X_valid = valid_df[features]
y_valid_log = np.log1p(valid_df["sales"])
y_valid_raw = valid_df["sales"].values

X_test = test_df[features]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)


# ============================================================
# Validation Model
# ============================================================

model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    boosting_type="gbdt",
    n_estimators=6000,
    learning_rate=0.015,
    num_leaves=96,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.15,
    reg_lambda=0.25,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid_log)],
    eval_metric="rmse",
    categorical_feature=cat_cols,
    callbacks=[
        lgb.early_stopping(stopping_rounds=250),
        lgb.log_evaluation(period=100),
    ],
)

valid_pred = np.expm1(model.predict(X_valid, num_iteration=model.best_iteration_))
valid_pred = np.maximum(valid_pred, 0)

score = rmsle(y_valid_raw, valid_pred)
print(f"Validation RMSLE: {score:.6f}")

best_iterations = model.best_iteration_
if best_iterations is None or best_iterations <= 0:
    best_iterations = 3500

print("Best iterations:", best_iterations)


# ============================================================
# Retrain On Full Training Data
# ============================================================

full_train_df = df[df["is_test"] == 0].copy()

X_full = full_train_df[features]
y_full = np.log1p(full_train_df["sales"])

final_model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    boosting_type="gbdt",
    n_estimators=best_iterations,
    learning_rate=0.015,
    num_leaves=96,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.15,
    reg_lambda=0.25,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

final_model.fit(
    X_full,
    y_full,
    categorical_feature=cat_cols,
)

test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.maximum(test_pred, 0)


# ============================================================
# Submission
# ============================================================

submission = pd.read_csv(SAMPLE_PATH)
submission["sales"] = test_pred
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Saved submission.csv")

Using data directory: /kaggle/input/competitions/store-sales-time-series-forecasting
Train shape: (3000888, 6)
Test shape: (28512, 5)
Max train date: 2017-08-15 00:00:00
Validation start: 2017-07-31 00:00:00
X_train: (2972376, 55)
X_valid: (28512, 55)
X_test: (28512, 55)
Training until validation scores don't improve for 250 rounds
[100]	valid_0's rmse: 0.742528
[200]	valid_0's rmse: 0.463228
[300]	valid_0's rmse: 0.425964
[400]	valid_0's rmse: 0.41651
[500]	valid_0's rmse: 0.412157
[600]	valid_0's rmse: 0.409512
[700]	valid_0's rmse: 0.406693
[800]	valid_0's rmse: 0.404349
[900]	valid_0's rmse: 0.402691
[1000]	valid_0's rmse: 0.401333
[1100]	valid_0's rmse: 0.400201
[1200]	valid_0's rmse: 0.399279
[1300]	valid_0's rmse: 0.398445
[1400]	valid_0's rmse: 0.397763
[1500]	valid_0's rmse: 0.397288
[1600]	valid_0's rmse: 0.396669
[1700]	valid_0's rmse: 0.395892
[1800]	valid_0's rmse: 0.395584
[1900]	valid_0's rmse: 0.395244
[2000]	valid_0's rmse: 0.394955
[2100]	valid_0's rmse: 0.394573
[220